# Introduction to CUDA Programming

### Instructor : Othmane K

This lab provides an introduction to CUDA programming, a parallel computing platform and API model developed by NVIDIA for general-purpose computing on GPUs. You will explore fundamental concepts such as:

*   **CUDA Kernel Execution**: Understanding how functions are launched on the GPU.
*   **Memory Management**: Allocating and transferring data between host (CPU) and device (GPU) memory.
*   **Thread Hierarchy**: Working with blocks and threads to parallelize computations.
*   **Vector Addition**: Implementing a basic parallel vector addition to demonstrate GPU acceleration.
*   **Matrix Multiplication**: Implementing parallel matrix multiplication.

By the end of this lab, you should have a solid understanding of the basics of CUDA programming and be able to implement simple parallel algorithms on a GPU.

## 1- Setup

Before we begin, we need to ensure our environment is correctly set up with the necessary tools for CUDA programming. This includes checking Python and NVCC versions and installing `nvcc4jupyter` to allow running CUDA C/C++ code directly in Colab notebooks.

In [ ]:
!python --version
!nvcc --version
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Python 3.12.13
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpnsruo8j2".


## 2- Your First CUDA Kernel: Simple Addition

Let's start with a very basic CUDA program to understand the fundamental concepts of launching a kernel on the GPU. This example demonstrates how to perform a simple addition of two integers on the GPU.

In [ ]:
%%cuda

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// ======================================================
// GPU KERNEL (runs on the GPU)
// This function executes on the device
// ======================================================
__global__ void add(int *a, int *b, int *c) {
    *c = *a + *b;   // GPU performs the addition
}

// ======================================================
// MAIN FUNCTION (runs on the CPU)
// This is HOST code
// ======================================================
int main(void) {

    // -------------------------------
    // 1️⃣ Host variables (CPU memory)
    // -------------------------------
    int a, b, c;              // host copies of a, b, c

    // -------------------------------
    // 2️⃣ Device pointers (GPU memory)
    // -------------------------------
    int *d_a, *d_b, *d_c;     // device copies of a, b, c

    int size = sizeof(int);

    // -------------------------------
    // 3️⃣ Allocate GPU memory
    // -------------------------------
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // -------------------------------
    // 4️⃣ Initialize host values
    // -------------------------------
    a = 2;
    b = 7;

    // -------------------------------
    // 5️⃣ Copy inputs CPU → GPU
    // -------------------------------
    cudaMemcpy(d_a, &a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, &b, size, cudaMemcpyHostToDevice);

    // -------------------------------
    // 6️⃣ Launch GPU kernel
    // <<<1,1>>> means:
    //   1 block, 1 thread
    // -------------------------------
    add<<<1,1>>>(d_a, d_b, d_c);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // -------------------------------
    // 7️⃣ Copy result GPU → CPU
    // -------------------------------
    cudaMemcpy(&c, d_c, size, cudaMemcpyDeviceToHost);

    // -------------------------------
    // 8️⃣ Print result (CPU)
    // -------------------------------
    cout << "a = " << a << endl;
    cout << "b = " << b << endl;
    cout << "c = a + b (computed on GPU) = " << c << endl;


}


a = 2
b = 7
c = a + b (computed on GPU) = 9



## 3- Parallel Vector Addition

Now, let's move to a more practical example: parallel vector addition. This section will demonstrate how to add two arrays (vectors) element-wise using the GPU, distributing the workload across multiple CUDA blocks where each block contains only one thread.

### 3.1- Parallel Vector Addition with N blocks (Each block 1 thread)

In [ ]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// ======================================================
// GPU KERNEL
// Each thread adds ONE element
// ======================================================
__global__ void add(int *a, int *b, int *c, int n) {

    // Compute global thread index
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Make sure we dont go out of bounds
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

// ======================================================
// MAIN (CPU code)
// ======================================================
int main() {

    int N = 124;                     // number of elements
    int size = N * sizeof(int);

    // -------------------------------
    // 1️⃣ Allocate host memory
    // -------------------------------
    int *h_a = new int[N];
    int *h_b = new int[N];
    int *h_c = new int[N];

    // Initialize host arrays
    for (int i = 0; i < N; i++) {
        h_a[i] = i;
        h_b[i] = i * 2;
    }

    // -------------------------------
    // 2️⃣ Allocate device memory
    // -------------------------------
    int *d_a, *d_b, *d_c;

    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    // -------------------------------
    // 3️⃣ Copy CPU → GPU
    // -------------------------------
    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    // -------------------------------
    // 4️⃣ Launch kernel
    // Well use 1 thread per block
    // -------------------------------
    int threadsPerBlock = 1;
    int blocks = (N + threadsPerBlock - 1) / threadsPerBlock; //just a formula

    add<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, N);

    cudaDeviceSynchronize();

    // -------------------------------
    // 5️⃣ Copy GPU → CPU
    // -------------------------------
    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    // -------------------------------
    // 6️⃣ Print first 10 results
    // -------------------------------
    cout << "First 10 results:\n";
    for (int i = 0; i < 10; i++) {
        cout << h_a[i] << " + " << h_b[i]
             << " = " << h_c[i] << endl;
    }

    // -------------------------------
    // 7️⃣ Cleanup
    // -------------------------------
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    delete[] h_a;
    delete[] h_b;
    delete[] h_c;

    return 0;
}


First 10 results:
0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
5 + 10 = 15
6 + 12 = 18
7 + 14 = 21
8 + 16 = 24
9 + 18 = 27



### Exercise: Vector Addition with Multiple Blocks and Threads

Your task is to modify the previous vector addition example to use a more flexible thread hierarchy. Implement vector addition where the work is distributed across 124 blocks, with each block containing 124 threads. Make sure to adjust the kernel launch configuration accordingly and handle potential out-of-bounds memory access.

### Task: Implement Vector Addition with 124 Blocks and 124 Threads

In [ ]:
# Here goes your code

## 4- Understanding Thread Indexing

In CUDA, understanding how to map global indices to threads and blocks is crucial for efficient parallel programming. This section introduces the concept of `threadIdx.x`, `blockIdx.x`, and `blockDim.x` which are essential for calculating a unique global index for each thread in a grid. We will use a 1D stencil computation as an example to illustrate these concepts.

### Indexing : Threads and Blocks

In [ ]:
%%cuda

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

#define N 16
#define RADIUS 1
#define BLOCK_SIZE 16

// ---------------- GPU KERNEL ----------------
__global__ void stencil_1d(int *in, int *out, int n) {

    int i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i >= n) return;

    int result = 0;

    // Apply 1D stencil with boundary check
    for (int offset = -RADIUS; offset <= RADIUS; offset++) {
        int index = i + offset;
        if (index >= 0 && index < n)
            result += in[index];
    }

    out[i] = result;
}

// ---------------- MAIN ----------------
int main() {

    int size = N * sizeof(int);

    int h_in[N];
    int h_out[N];

    int *d_in, *d_out;

    // Initialize input
    for (int i = 0; i < N; i++)
        h_in[i] = i + 1;   // 1,2,3,...16

    // Allocate device memory
    cudaMalloc(&d_in, size);
    cudaMalloc(&d_out, size);

    // Copy to GPU
    cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice);

    // Launch kernel
    stencil_1d<<<1, BLOCK_SIZE>>>(d_in, d_out, N);

    // Copy back
    cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost);

    // Print results
    cout << "Input:  ";
    for (int i = 0; i < N; i++)
        cout << h_in[i] << " ";
    cout << endl;

    cout << "Output: ";
    for (int i = 0; i < N; i++)
        cout << h_out[i] << " ";
    cout << endl;

    cudaFree(d_in);
    cudaFree(d_out);

    return 0;
}


Input:  1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 
Output: 3 6 9 12 15 18 21 24 27 30 33 36 39 42 45 31 



## 5- Exercise: CUDA Hello World with Multiple Blocks and Threads

Write a CUDA kernel that prints "Hello Udemy" along with the `blockIdx.x` and `threadIdx.x` for each thread. Configure the kernel to launch with 2 blocks and 2 threads per block. This exercise will help you grasp how threads and blocks are identified within a CUDA grid.

### Task: Implement CUDA Hello World with 2 Blocks and 2 Threads

In [ ]:
%%cuda
#include <stdio.h>
__global__ void hello(){

                       printf("Hello Udemy from block : %u, thread : %u \n", blockIdx.x, threadIdx.x );
}

int main(){

           hello<<<,>>>();
           cudaDeviceSynchronize();
           return 0;
}

Hello IA 2A from block : 1, thread : 0 
Hello IA 2A from block : 1, thread : 1 
Hello IA 2A from block : 0, thread : 0 
Hello IA 2A from block : 0, thread : 1 



## 6- Querying CUDA Device Properties

It's often useful to know the properties of the GPU device your CUDA code is running on, such as the number of available devices. This section demonstrates how to query the number of CUDA-enabled devices on your system using `cudaGetDeviceCount`.

### Getting the number of devices

In [ ]:
%%cuda

#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int deviceCount;
    cudaGetDeviceCount(&deviceCount);
    printf("Number of CUDA devices: %d\n", deviceCount);
    return 0;
}


Number of CUDA devices: 1



## 7- Vector Addition Revisited: Performance Considerations

We'll re-examine vector addition, first with a naive approach, and then progressively optimize it by properly utilizing the GPU's parallel capabilities. This will highlight the performance benefits of correct thread and block configuration.

### 7.0- Naive Vector Addition (Loop in Kernel)

In [ ]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <assert.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define N 10000000
#define MAX_ERR 1e-6

__global__ void vector_add(float *out, float *a, float *b, int n) {
    for(int i = 0; i < n; i ++){
        out[i] = a[i] + b[i];
    }
}

int main(){
    float *a, *b, *out;
    float *d_a, *d_b, *d_out;

    // Allocate host memory
    a   = (float*)malloc(sizeof(float) * N);
    b   = (float*)malloc(sizeof(float) * N);
    out = (float*)malloc(sizeof(float) * N);

    // Initialize host arrays
    for(int i = 0; i < N; i++){
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    // Allocate device memory
    cudaMalloc((void**)&d_a, sizeof(float) * N);
    cudaMalloc((void**)&d_b, sizeof(float) * N);
    cudaMalloc((void**)&d_out, sizeof(float) * N);

    // Transfer data from host to device memory
    cudaMemcpy(d_a, a, sizeof(float) * N, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, sizeof(float) * N, cudaMemcpyHostToDevice);

    // Executing kernel
    vector_add<<<1,1>>>(d_out, d_a, d_b, N);

    // Transfer data back to host memory
    cudaMemcpy(out, d_out, sizeof(float) * N, cudaMemcpyDeviceToHost);

    // Verification
    for(int i = 0; i < N; i++){
        assert(fabs(out[i] - a[i] - b[i]) < MAX_ERR);
    }
    printf("out[0] = %f\n", out[0]);
    printf("PASSED\n");

    // Deallocate device memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_out);

    // Deallocate host memory
    free(a);
    free(b);
    free(out);
}


out[0] = 3.000000
PASSED



### 7.1- Parallel Vector Addition with Proper Indexing and Timing

This improved version of vector addition correctly utilizes `blockIdx.x` and `threadIdx.x` to assign each element's computation to a unique thread. It also includes timing measurements to demonstrate the performance gain from parallel execution on the GPU.

In [ ]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <assert.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define N 10000000
#define MAX_ERR 1e-6

__global__ void vector_add(float *out, float *a, float *b, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        out[idx] = a[idx] + b[idx];
    }
}

int main() {
    float *a, *b, *out;
    float *d_a, *d_b, *d_out;

    // Allocate host memory
    a   = (float*)malloc(sizeof(float) * N);
    b   = (float*)malloc(sizeof(float) * N);
    out = (float*)malloc(sizeof(float) * N);

    // Initialize host arrays
    for(int i = 0; i < N; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    // Allocate device memory
    cudaMalloc((void**)&d_a, sizeof(float) * N);
    cudaMalloc((void**)&d_b, sizeof(float) * N);
    cudaMalloc((void**)&d_out, sizeof(float) * N);

    // Transfer data from host to device memory
    cudaMemcpy(d_a, a, sizeof(float) * N, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, sizeof(float) * N, cudaMemcpyHostToDevice);

    // Timing variables
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel launch parameters
    int blockSize = 1024;  // Number of threads per block
    int gridSize = (N + blockSize - 1) / blockSize;  // Number of blocks in the grid

    // Record the start event
    cudaEventRecord(start);

    // Executing kernel
    vector_add<<<gridSize, blockSize>>>(d_out, d_a, d_b, N);

    // Record the stop event
    cudaEventRecord(stop);

    // Wait for the stop event to complete
    cudaEventSynchronize(stop);

    // Calculate the elapsed time in milliseconds
    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    // Transfer data back to host memory
    cudaMemcpy(out, d_out, sizeof(float) * N, cudaMemcpyDeviceToHost);

    // Verification
    for(int i = 0; i < N; i++) {
        assert(fabs(out[i] - a[i] - b[i]) < MAX_ERR);
    }
    printf("out[0] = %f\n", out[0]);
    printf("PASSED\n");

    // Print the execution time
    printf("Execution time: %f ms\n", milliseconds);

    // Deallocate device memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_out);

    // Deallocate host memory
    free(a);
    free(b);
    free(out);

    return 0;
}


out[0] = 3.000000
PASSED
Execution time: 0.612896 ms



### 7.2- Vector Addition with Grid Calculation

Complete the `vector_add` kernel by correctly calculating the global thread ID (`tid`) using `blockIdx.x`, `blockDim.x`, and `threadIdx.x`. This will ensure that all elements of the vectors are processed in parallel across the entire grid.

### Task: Complete Vector Addition with Grid Calculation

In [ ]:
%%cuda

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <assert.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define N 10000000
#define MAX_ERR 1e-6

__global__ void vector_add(float *out, float *a, float *b, int n) {

    int tid = blockIdx.x...

    // Handling arbitrary vector size
    if (tid < n){
        out[tid] = a[tid] + b[tid];
    }
}

int main(){
    float *a, *b, *out;
    float *d_a, *d_b, *d_out;

    // Allocate host memory
    a   = (float*)malloc(sizeof(float) * N);
    b   = (float*)malloc(sizeof(float) * N);
    out = (float*)malloc(sizeof(float) * N);

    // Initialize host arrays
    for(int i = 0; i < N; i++){
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    // Allocate device memory
    cudaMalloc((void**)&d_a, sizeof(float) * N);
    cudaMalloc((void**)&d_b, sizeof(float) * N);
    cudaMalloc((void**)&d_out, sizeof(float) * N);

    // Transfer data from host to device memory
    cudaMemcpy(d_a, a, sizeof(float) * N, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, sizeof(float) * N, cudaMemcpyHostToDevice);


    // Executing kernel
    int block_size = 256;
    int grid_size = ((N + block_size) / block_size);
    vector_add<<<grid_size,block_size>>>(d_out, d_a, d_b, N);

    // Transfer data back to host memory
    cudaMemcpy(out, d_out, sizeof(float) * N, cudaMemcpyDeviceToHost);

    // Verification
    for(int i = 0; i < N; i++){
        assert(fabs(out[i] - a[i] - b[i]) < MAX_ERR);
    }

    printf("PASSED\n");

    // Deallocate device memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_out);

    // Deallocate host memory
    free(a);
    free(b);
    free(out);
}


PASSED



## 8- Matrix Multiplication Exercise

Implement a CUDA kernel for matrix multiplication. Your kernel should take two matrices `A` and `B` as input and compute their product `C`. Each thread should be responsible for calculating one element of the resulting matrix `C`. Consider the dimensions of the matrices and how to map thread indices to matrix coordinates effectively.

**Hint**: Refer to the previous sections on thread indexing to determine how to calculate the row and column for each thread in the output matrix `C`.

## Matrix multiplication


Matrix Multiplication:
--------------------------
Given two matrices 'A' and 'B' of dimensions 'm x n' and 'n x p' respectively,
their product is a new matrix 'C' of dimension 'm x p' where each element is defined as:

Cij = Ai1 * B1j + Ai2 * B2j + ... + Ain * Bnj

Geometrically, matrix multiplication can be seen as a linear transformation applied to the vectors in the space.  
It's useful for tasks like transforming coordinates, solving systems of linear equations,
and performing rotations and scaling in computer graphics.
Matrix multiplication is associative and distributive, but not commutative.


CUDA Tweaks:
--------------------
1. Thread Assignment: Each thread is assigned to compute one element of the resulting matrix.
   The index for each thread is calculated based on the block size, block index, and thread index within the block
   (e.g., thread (i, j) computes Cij).

2. Element Calculation: Each thread calculates the corresponding element in the resulting matrix
   by multiplying the elements in the corresponding row of the first matrix and the corresponding column
   of the second matrix, and summing up these products.

The use of atomic operations is not required in this case as each thread is responsible for calculating
and storing a unique element in the resulting matrix, avoiding race conditions.
It is tempting to consider using launching vectorDot kernels for each element of the resulting matrix,
This is not recommended as it would lead to a large number of kernel launches and resource overheads.


#### Task : Write a Cuda kernel to execute matrix multiplication
